# TB Portals â€” 03 Â· Train baseline (Day-1 GATE)

Reproduce Kantipudi A2: single DenseNet121 ALP regressor + cavity classifier, NAdam 1e-3, country-segregated. **Enable Internet** in Kaggle (DenseNet121 ImageNet weights download once).

Strategy on a T4 (9â€“12h sessions): run **1 seed Ã— 3 countries first** to hit the gate, then add seeds. Training appends to `results.csv` and is `--resume`-able, so you can split across sessions.

In [ ]:
# ── Pull latest codebase from GitHub ─────────────────────────────────────
import os, subprocess, sys

REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("repo ready at", REPO_DIR)


In [ ]:
# --- Clone / update the repo (requires Internet enabled in Kaggle) ---
import os, subprocess, sys

REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"

if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("repo ready at", REPO_DIR)

In [ ]:
import sys, os
REPO_DIR = '/kaggle/working/dl-project-codebase'
WORK     = '/kaggle/working'
MANIFEST = f'{WORK}/data/processed/tbportals_manifest.csv'
OUT_DIR  = f'{WORK}/checkpoints/tbportals/baseline'
os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, REPO_DIR)
from src.training.train_tbportals_baseline import main as train_main
print('manifest:', MANIFEST)
print('out_dir: ', OUT_DIR)


### Smoke run (confirm it trains) â€” 1 country, 1 seed, 3 epochs

In [ ]:
# Multi-seed run — seeds 1 and 2 (run after gate passes)
train_main([
    '--manifest',    MANIFEST,
    '--held-outs',   'Romania', 'Moldova', 'Kazakhstan',
    '--seeds',       '1', '2',
    '--epochs',      '50',
    '--batch-size',  '32',
    '--num-workers', '2',
    '--amp',
    '--out-dir',     OUT_DIR,
])


### Gate run â€” 3 countries, seed 0, 30 epochs
After notebook 02 you can add: `'--use-lung-crop', '--crops-dir', f'{WORK}/data/processed/tbportals_crops'`.

In [ ]:
# Gate run — 3 held-out countries, seed 0, 50 epochs
# Kaggle P100/T4: ~90 min total
train_main([
    '--manifest',    MANIFEST,
    '--held-outs',   'Romania', 'Moldova', 'Kazakhstan',
    '--seeds',       '0',
    '--epochs',      '50',
    '--batch-size',  '32',
    '--num-workers', '2',
    '--amp',
    '--out-dir',     OUT_DIR,
])


### Add seeds for meanÂ±std (run when you have session time)

In [ ]:
train_main([
    '--manifest', MANIFEST,
    '--held-outs', 'Romania', 'Moldova', 'Kazakhstan',
    '--seeds', '1', '2',
    '--epochs', '30',
    '--out-dir', OUT_DIR,
])